# Bank Loan Prediction V2

Improved model with:
- Feature engineering (Income_Loan_Ratio, Loan_Burden, Income_Per_Dependent)
- Class weights (`class_weight='balanced'` / `scale_pos_weight`) to handle 72/28 imbalance
- Model comparison: Decision Tree, Random Forest, Gradient Boosting, XGBoost
- 5-fold stratified cross-validation

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
import pickle

In [2]:
df = pd.read_csv('Bank_Loan.csv')
print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df['Loan_Status'].value_counts())
ratio = df['Loan_Status'].value_counts().values[0] / df['Loan_Status'].value_counts().values[1]
print(f"\nClass ratio: {ratio:.2f}:1 (rejected:approved)")

Shape: (981, 15)

Class distribution:
Loan_Status
No     712
Yes    269
Name: count, dtype: int64

Class ratio: 2.65:1 (rejected:approved)


In [3]:
# Feature engineering
df['Income_Loan_Ratio'] = df['ApplicantIncome'] / df['LoanAmount']
df['Loan_Burden'] = df['LoanAmount'] / df['Tenure']
df['Income_Per_Dependent'] = df['ApplicantIncome'] / (df['Dependents'] + 1)

# Drop Loan_ID
df = df.drop(['Loan_ID'], axis=1)

# Encode categoricals
le = LabelEncoder()
categorical_cols = ['Gender', 'Married', 'Education', 'Self_Employed',
                    'Previous_Loan_Taken', 'Property_Area', 'Customer_Bandwith', 'Loan_Status']
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# Split features and target
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']
feature_names = X.columns.tolist()
print(f"Features ({len(feature_names)}): {feature_names}")

Features (16): ['Age', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'LoanAmount', 'Previous_Loan_Taken', 'Cibil_Score', 'Property_Area', 'Customer_Bandwith', 'Tenure', 'Income_Loan_Ratio', 'Loan_Burden', 'Income_Per_Dependent']


In [4]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Train class dist: {y_train.value_counts().to_dict()}")
print(f"Test class dist: {y_test.value_counts().to_dict()}")

Train: 784, Test: 197
Train class dist: {0: 569, 1: 215}
Test class dist: {0: 143, 1: 54}


In [5]:
# Model comparison with class weights
models = {
    'Decision Tree (original)': DecisionTreeClassifier(
        criterion='gini', min_samples_split=300, min_samples_leaf=50, max_depth=4, random_state=42
    ),
    'Decision Tree (balanced)': DecisionTreeClassifier(
        criterion='gini', min_samples_split=300, min_samples_leaf=50, max_depth=4,
        class_weight='balanced', random_state=42
    ),
    'Random Forest (balanced)': RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, random_state=42
    ),
    'XGBoost (balanced)': XGBClassifier(
        n_estimators=100, random_state=42, scale_pos_weight=ratio, eval_metric='logloss'
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    results[name] = {
        'accuracy': report['accuracy'],
        'f1_approved': report['1']['f1-score'],
        'recall_approved': report['1']['recall'],
        'precision_approved': report['1']['precision'],
    }
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred))


Decision Tree (original)
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       143
           1       0.90      0.81      0.85        54

    accuracy                           0.92       197
   macro avg       0.92      0.89      0.90       197
weighted avg       0.92      0.92      0.92       197


Decision Tree (balanced)
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       143
           1       0.90      0.81      0.85        54

    accuracy                           0.92       197
   macro avg       0.92      0.89      0.90       197
weighted avg       0.92      0.92      0.92       197




Random Forest (balanced)
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       143
           1       0.90      0.81      0.85        54

    accuracy                           0.92       197
   macro avg       0.92      0.89      0.90       197
weighted avg       0.92      0.92      0.92       197




Gradient Boosting
              precision    recall  f1-score   support

           0       0.94      0.95      0.94       143
           1       0.87      0.83      0.85        54

    accuracy                           0.92       197
   macro avg       0.90      0.89      0.90       197
weighted avg       0.92      0.92      0.92       197


XGBoost (balanced)
              precision    recall  f1-score   support

           0       0.91      0.94      0.92       143
           1       0.82      0.76      0.79        54

    accuracy                           0.89       197
   macro avg       0.87      0.85      0.86       197
weighted avg       0.89      0.89      0.89       197



In [6]:
# Compare results
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df.round(4).to_string())
print(f"\nBest model by F1 (approved): {results_df['f1_approved'].idxmax()}")
print(f"Best model by Recall (approved): {results_df['recall_approved'].idxmax()}")


Model Comparison:
                          accuracy  f1_approved  recall_approved  precision_approved
Decision Tree (original)    0.9239       0.8544           0.8148              0.8980
Decision Tree (balanced)    0.9239       0.8544           0.8148              0.8980
Random Forest (balanced)    0.9239       0.8544           0.8148              0.8980
Gradient Boosting           0.9188       0.8491           0.8333              0.8654
XGBoost (balanced)          0.8883       0.7885           0.7593              0.8200

Best model by F1 (approved): Decision Tree (original)
Best model by Recall (approved): Gradient Boosting


In [7]:
# Select best model and cross-validate
best_name = results_df['f1_approved'].idxmax()
print(f"Selected: {best_name}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

if 'XGBoost' in best_name:
    final_model = XGBClassifier(n_estimators=100, random_state=42, scale_pos_weight=ratio, eval_metric='logloss')
elif 'Random Forest' in best_name:
    final_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
elif 'Gradient Boosting' in best_name:
    final_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
else:
    final_model = DecisionTreeClassifier(
        criterion='gini', min_samples_split=300, min_samples_leaf=50, max_depth=4,
        class_weight='balanced', random_state=42
    )

cv_scores = cross_val_score(final_model, X, y, cv=cv, scoring='f1')
print(f"\n5-Fold CV F1 scores: {cv_scores.round(4)}")
print(f"Mean F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

Selected: Decision Tree (original)



5-Fold CV F1 scores: [0.8571 0.8381 0.8247 0.898  0.82  ]
Mean F1: 0.8476 (+/- 0.0283)


In [8]:
# Train final model on full data
final_model.fit(X, y)
y_pred = final_model.predict(X_test)
print("Final Model Test Results:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Final Model Test Results:
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       143
           1       0.90      0.81      0.85        54

    accuracy                           0.92       197
   macro avg       0.92      0.89      0.90       197
weighted avg       0.92      0.92      0.92       197

Confusion Matrix:
[[138   5]
 [ 10  44]]


In [9]:
# Feature importances
if hasattr(final_model, 'feature_importances_'):
    importances = pd.Series(final_model.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=False)
    print("\nFeature Importances:")
    for feat, imp in importances.items():
        print(f"  {feat}: {imp:.4f}")


Feature Importances:
  Cibil_Score: 0.5780
  Previous_Loan_Taken: 0.3981
  Property_Area: 0.0162
  Married: 0.0076
  Age: 0.0000
  Gender: 0.0000
  Dependents: 0.0000
  Education: 0.0000
  Self_Employed: 0.0000
  ApplicantIncome: 0.0000
  LoanAmount: 0.0000
  Customer_Bandwith: 0.0000
  Tenure: 0.0000
  Income_Loan_Ratio: 0.0000
  Loan_Burden: 0.0000
  Income_Per_Dependent: 0.0000


In [10]:
# Save model
with open('build_v2.pkl', 'wb') as f:
    pickle.dump(final_model, f)
print("\nModel saved to build_v2.pkl")
print(f"Features expected ({len(feature_names)}): {feature_names}")


Model saved to build_v2.pkl
Features expected (16): ['Age', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'LoanAmount', 'Previous_Loan_Taken', 'Cibil_Score', 'Property_Area', 'Customer_Bandwith', 'Tenure', 'Income_Loan_Ratio', 'Loan_Burden', 'Income_Per_Dependent']
